# D3 — Evaluation + Ablation (Member 3)
### Part 1: Gold Q/A Set + Faithfulness / Answer-Relevance Judge

This notebook builds the two reusable pieces the rest of the D3 evaluation depends on:

1. A **gold Q/A set** — a curated subset of the existing labeled queries, with their silver `relevant_paper_ids` exposed for a quick human review pass.
2. **Faithfulness** and **answer relevance** scoring functions, used as an LLM-judge (Claude, reference-free — judges the answer against its own cited evidence / against the question, no hand-written reference answer required) with a TF-IDF heuristic fallback if no API key is set.

Citation coverage and latency don't need anything new — they're already emitted per-run by Member 1's `run_graphrag_executor()` / `ask()` (`quality_checks["citation_coverage"]`, `timing["total_ms"]`). This notebook only fills the gap: faithfulness and relevance, which need either a judge or a gold reference.

Nothing here changes what the system can answer — the GraphRAG executor still answers any question at run time via retrieval. The gold set is only used afterward, to grade what it produced.

In [1]:
from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path("..").resolve()
CACHE_DIR = PROJECT_ROOT / "notebook_cache"

INGESTION_DIR = CACHE_DIR / "01_ingestion"
EVAL_ABLATION_CACHE = CACHE_DIR / "06_eval_ablation"
EVAL_ABLATION_CACHE.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Eval/ablation cache:", EVAL_ABLATION_CACHE)

Project root: C:\Users\97150\Documents\GitHub\CLPapersAIAgent
Eval/ablation cache: C:\Users\97150\Documents\GitHub\CLPapersAIAgent\notebook_cache\06_eval_ablation


## 1. Load the existing labeled queries + paper metadata

In [2]:
required_files = {
    "queries": INGESTION_DIR / "labeled_query_set.json",
    "metadata": INGESTION_DIR / "metadata.json",
}

verification_df = pd.DataFrame([
    {"name": name, "path": str(path), "exists": path.exists()}
    for name, path in required_files.items()
])
display(verification_df)

missing = verification_df[verification_df["exists"] == False]
if len(missing) > 0:
    raise FileNotFoundError(
        "Missing required prior outputs. Run 01_ingestion_pipeline.ipynb first "
        "to produce labeled_query_set.json and metadata.json."
    )

with open(required_files["queries"], "r", encoding="utf-8") as f:
    all_queries = json.load(f)

with open(required_files["metadata"], "r", encoding="utf-8") as f:
    metadata = json.load(f)

metadata_lookup = {m["paper_id"]: m for m in metadata if "paper_id" in m}

print(f"Loaded {len(all_queries)} queries, {len(metadata_lookup)} papers in metadata.")

,name,path,exists
0,queries,C:\Users\97150\Documents\GitHub\CLPapersAIAgent\notebook_cache\01_ingestion\labeled_query_set.json,True
1,metadata,C:\Users\97150\Documents\GitHub\CLPapersAIAgent\notebook_cache\01_ingestion\metadata.json,True


Loaded 20 queries, 208 papers in metadata.


## 2. Select a gold subset (10-15 questions, stratified by category)

Categories whose silver labeling actually found relevant papers get priority — a question with
zero `relevant_paper_ids` gives nothing to grade citations against. If there are more categories
than the target size, the categories with the strongest silver matches win the slots; if there
are fewer categories than the target size, the remaining slots are topped up with the next-best
labeled queries regardless of category.

In [3]:
TARGET_GOLD_SIZE = 12  # keep this in the 10-15 range

by_category = {}
for q in all_queries:
    by_category.setdefault(q.get("category", "uncategorized"), []).append(q)

categories_sorted = sorted(
    by_category.items(),
    key=lambda kv: max((q.get("num_relevant", 0) for q in kv[1]), default=0),
    reverse=True,
)

selected = []
selected_ids = set()

for category, qs in categories_sorted:
    if len(selected) >= TARGET_GOLD_SIZE:
        break
    best = max(qs, key=lambda q: q.get("num_relevant", 0))
    selected.append(best)
    selected_ids.add(best["query_id"])

if len(selected) < TARGET_GOLD_SIZE:
    remaining = [q for q in all_queries if q["query_id"] not in selected_ids]
    remaining_sorted = sorted(remaining, key=lambda q: q.get("num_relevant", 0), reverse=True)
    selected.extend(remaining_sorted[: TARGET_GOLD_SIZE - len(selected)])

n_categories = len(set(q.get("category", "uncategorized") for q in selected))
print(f"Selected {len(selected)} of {len(all_queries)} queries across {n_categories} categories.")
display(pd.DataFrame(selected)[["query_id", "category", "text", "num_relevant"]])

Selected 12 of 20 queries across 12 categories.


,query_id,category,text,num_relevant
0,q002,pretraining,large language model pre-training on multilingual corpora,65
1,q007,evaluation,benchmark evaluation of LLMs on commonsense reasoning,39
2,q005,NER,named entity recognition in low-resource languages,25
3,q006,reasoning,chain-of-thought prompting for mathematical reasoning,24
4,q013,agents,LLM-based agents with tool use and function calling,16
5,q009,efficiency,parameter efficient fine-tuning with LoRA and adapter layers,14
6,q004,RAG,retrieval augmented generation for open-domain question answering,11
7,q011,multimodal,vision language models for image captioning and VQA,11
8,q016,safety,jailbreak attacks and safety evaluation of LLMs,11
9,q008,hallucination,hallucination detection and factual consistency in generation,10


## 3. Export for a quick human review pass

This is the only manual step. The CSV below shows each selected question next to the paper
titles the silver keyword-matcher (in `create_query_set.py`) already found for it. Open the CSV,
skim `silver_paper_titles`, and:

- if the silver matches look right, leave `reviewed_paper_ids` **blank** — the loader below will
  just keep the silver labels
- if they're wrong or incomplete, type the correct comma-separated `paper_id`s into
  `reviewed_paper_ids`
- optionally jot a one-line note in `expected_answer_notes` (e.g. "should mention X and Y") —
  useful context later, not required

Save the file in place and re-run the loader cell in Section 4.

In [4]:
def format_paper_list(paper_ids):
    parts = []
    for pid in paper_ids:
        title = metadata_lookup.get(pid, {}).get("title", "unknown title")
        parts.append(f"{pid}: {title}")
    return " | ".join(parts) if parts else "(none -- silver matching found nothing)"

review_rows = []
for q in selected:
    review_rows.append({
        "query_id": q["query_id"],
        "category": q.get("category", ""),
        "question": q["text"],
        "silver_paper_ids": ", ".join(q.get("relevant_paper_ids", [])),
        "silver_paper_titles": format_paper_list(q.get("relevant_paper_ids", [])),
        "reviewed_paper_ids": "",       # fill in by hand; leave blank to accept silver as-is
        "expected_answer_notes": "",    # optional
    })

review_df = pd.DataFrame(review_rows)
review_path = EVAL_ABLATION_CACHE / "gold_set_for_review.csv"
review_df.to_csv(review_path, index=False)

print(f"Saved {len(review_df)} questions for manual review -> {review_path}")
display(review_df[["query_id", "category", "question", "silver_paper_titles"]])

Saved 12 questions for manual review -> C:\Users\97150\Documents\GitHub\CLPapersAIAgent\notebook_cache\06_eval_ablation\gold_set_for_review.csv


,query_id,category,question,silver_paper_titles
0,q002,pretraining,large language model pre-training on multilingual corpora,paper4: mdok-style at SemEval-2026 Task 9: Finetuning LLMs for Multilingual Polarization Detection | paper5: Fuzzy F...
1,q007,evaluation,benchmark evaluation of LLMs on commonsense reasoning,"paper8: Synthetic Users, Real Differences: an Evaluation Framework for User Simulation in Multi-Turn Conversations |..."
2,q005,NER,named entity recognition in low-resource languages,paper10: Dependency Parsing Across the Resource Spectrum: Evaluating Architectures on High and Low-Resource Language...
3,q006,reasoning,chain-of-thought prompting for mathematical reasoning,paper15: Accurate Legal Reasoning at Scale: Neuro-Symbolic Offloading and Structural Auditability for Robust Legal A...
4,q013,agents,LLM-based agents with tool use and function calling,paper1: FlexSQL: Flexible Exploration and Execution Make Better Text-to-SQL Agents | paper2: Reinforcement Learning ...
5,q009,efficiency,parameter efficient fine-tuning with LoRA and adapter layers,paper1: FlexSQL: Flexible Exploration and Execution Make Better Text-to-SQL Agents | paper54: BIM Information Extrac...
6,q004,RAG,retrieval augmented generation for open-domain question answering,paper6: ContextualJailbreak: Evolutionary Red-Teaming via Simulated Conversational Priming | paper12: Generation: A ...
7,q011,multimodal,vision language models for image captioning and VQA,paper18: PC-MNet: Dual-Level Congruity Modeling for Multimodal Sarcasm Detection via Polarity-Modulated Attention | ...
8,q016,safety,jailbreak attacks and safety evaluation of LLMs,paper6: ContextualJailbreak: Evolutionary Red-Teaming via Simulated Conversational Priming | paper31: ARGUS: Policy-...
9,q008,hallucination,hallucination detection and factual consistency in generation,paper14: A multilingual hallucination benchmark: MultiWikiQHalluA | paper19: HalluScan: A Systematic Benchmark for D...


## 4. Load the (possibly edited) review CSV into the final gold Q/A set

In [5]:
reviewed_path = EVAL_ABLATION_CACHE / "gold_set_for_review.csv"  # edit by hand, then re-run this cell
reviewed_df = pd.read_csv(reviewed_path, keep_default_na=False)

gold_qa_set = []
for _, row in reviewed_df.iterrows():
    reviewed = [p.strip() for p in str(row["reviewed_paper_ids"]).split(",") if p.strip()]
    silver = [p.strip() for p in str(row["silver_paper_ids"]).split(",") if p.strip()]
    final_ids = reviewed if reviewed else silver

    gold_qa_set.append({
        "question_id": row["query_id"],
        "category": row["category"],
        "question": row["question"],
        "gold_relevant_paper_ids": final_ids,
        "expected_answer_notes": row.get("expected_answer_notes", "") or "",
        "source": "human_reviewed" if reviewed else "silver_unreviewed",
    })

gold_set_path = EVAL_ABLATION_CACHE / "gold_qa_set.json"
with open(gold_set_path, "w", encoding="utf-8") as f:
    json.dump(gold_qa_set, f, indent=2, ensure_ascii=False)

reviewed_count = sum(1 for g in gold_qa_set if g["source"] == "human_reviewed")
print(f"Saved {len(gold_qa_set)} questions -> {gold_set_path}")
print(f"{reviewed_count} reviewed by hand, {len(gold_qa_set) - reviewed_count} still on silver labels.")
display(pd.DataFrame(gold_qa_set)[["question_id", "category", "question", "gold_relevant_paper_ids", "source"]])

Saved 12 questions -> C:\Users\97150\Documents\GitHub\CLPapersAIAgent\notebook_cache\06_eval_ablation\gold_qa_set.json
0 reviewed by hand, 12 still on silver labels.


,question_id,category,question,gold_relevant_paper_ids,source
0,q002,pretraining,large language model pre-training on multilingual corpora,"[paper4, paper5, paper14, paper22, paper23, paper29, paper37, paper39, paper42, paper43, paper44, paper45, paper46, ...",silver_unreviewed
1,q007,evaluation,benchmark evaluation of LLMs on commonsense reasoning,"[paper8, paper12, paper14, paper15, paper19, paper28, paper34, paper42, paper43, paper44, paper47, paper53, paper56,...",silver_unreviewed
2,q005,NER,named entity recognition in low-resource languages,"[paper10, paper12, paper25, paper40, paper45, paper47, paper49, paper54, paper59, paper60, paper63, paper64, paper68...",silver_unreviewed
3,q006,reasoning,chain-of-thought prompting for mathematical reasoning,"[paper15, paper34, paper42, paper43, paper44, paper53, paper63, paper68, paper70, paper103, paper112, paper113, pape...",silver_unreviewed
4,q013,agents,LLM-based agents with tool use and function calling,"[paper1, paper2, paper9, paper40, paper61, paper71, paper78, paper111, paper122, paper126, paper127, paper128, paper...",silver_unreviewed
5,q009,efficiency,parameter efficient fine-tuning with LoRA and adapter layers,"[paper1, paper54, paper59, paper60, paper87, paper96, paper120, paper123, paper160, paper177, paper183, paper184, pa...",silver_unreviewed
6,q004,RAG,retrieval augmented generation for open-domain question answering,"[paper6, paper12, paper17, paper63, paper85, paper107, paper119, paper132, paper148, paper169, paper187]",silver_unreviewed
7,q011,multimodal,vision language models for image captioning and VQA,"[paper18, paper38, paper80, paper107, paper110, paper138, paper165, paper181, paper201, paper205, paper206]",silver_unreviewed
8,q016,safety,jailbreak attacks and safety evaluation of LLMs,"[paper6, paper31, paper56, paper114, paper115, paper123, paper125, paper126, paper136, paper144, paper146]",silver_unreviewed
9,q008,hallucination,hallucination detection and factual consistency in generation,"[paper14, paper19, paper27, paper33, paper49, paper67, paper85, paper97, paper135, paper149]",silver_unreviewed


## 5. Faithfulness + Answer Relevance judge

Both metrics are **reference-free** — no hand-written answer is needed.

- **Faithfulness**: feed the judge the generated answer + the evidence it was built from, ask
  whether every claim is actually supported by that evidence.
- **Answer relevance**: feed the judge the question + the generated answer, ask whether it
  actually addresses the question.

If `ANTHROPIC_API_KEY` is set and the `anthropic` package is installed, these use Claude
(Haiku, since it's a cheap structured-rating task) as the judge. Otherwise they fall back to a
TF-IDF cosine-similarity heuristic. Note the heuristic is a rough proxy for faithfulness
specifically: since `generate_simple_answer()` largely re-states the evidence text, TF-IDF
overlap will read as artificially high regardless of mode — it's useful as a sanity check or
when no API key is available, but the LLM judge is what should drive the final numbers.

In [6]:
import os

JUDGE_MODEL = "claude-haiku-4-5-20251001"  # cheap/fast model, plenty for a 1-5 rating task
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

try:
    import anthropic
    _client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None
except ImportError:
    anthropic = None
    _client = None

USE_LLM_JUDGE = _client is not None

print("LLM judge available:", USE_LLM_JUDGE)
if not USE_LLM_JUDGE:
    print("No ANTHROPIC_API_KEY found (or `anthropic` package missing) -> falling back to TF-IDF heuristic.")
    print("To enable the LLM judge: pip install anthropic, then set ANTHROPIC_API_KEY in your environment.")

LLM judge available: False
No ANTHROPIC_API_KEY found (or `anthropic` package missing) -> falling back to TF-IDF heuristic.
To enable the LLM judge: pip install anthropic, then set ANTHROPIC_API_KEY in your environment.


In [8]:
import re as _re

def _call_judge(prompt, max_tokens=300):
    response = _client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(block.text for block in response.content if block.type == "text")


def _parse_score(text, default=0.0):
    """Pull the first 1-5 rating out of the judge's reply."""
    match = _re.search(r"\b([1-5])\b", text)
    return float(match.group(1)) if match else default


FAITHFULNESS_PROMPT = """You are grading a retrieval-augmented answer for FAITHFULNESS.

Question: {question}

Answer to grade:
{answer}

Evidence the answer is supposed to be grounded in:
{evidence}

Rate from 1 to 5 how well every claim in the answer is supported by the evidence above:
5 = every claim is directly supported by the evidence, no unsupported additions
3 = mostly supported, with at least one unsupported or stretched claim
1 = largely unsupported or contradicts the evidence

Reply with the score as a single digit first, then one short sentence of justification."""

RELEVANCE_PROMPT = """You are grading a retrieval-augmented answer for ANSWER RELEVANCE.

Question: {question}

Answer to grade:
{answer}

Rate from 1 to 5 how directly the answer addresses the question asked (ignore whether it is
correct -- only judge whether it is on-topic and responsive):
5 = directly and completely addresses the question
3 = partially addresses it, or includes a lot of irrelevant material
1 = does not address the question at all

Reply with the score as a single digit first, then one short sentence of justification."""


def judge_faithfulness_llm(question, answer, evidence_snippets):
    evidence_text = "\n".join(f"- {e}" for e in evidence_snippets) or "(no evidence retrieved)"
    prompt = FAITHFULNESS_PROMPT.format(question=question, answer=answer, evidence=evidence_text)
    text = _call_judge(prompt)
    return {"score": _parse_score(text), "raw": text, "method": "llm_judge"}


def judge_answer_relevance_llm(question, answer):
    prompt = RELEVANCE_PROMPT.format(question=question, answer=answer)
    text = _call_judge(prompt)
    return {"score": _parse_score(text), "raw": text, "method": "llm_judge"}

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def _tfidf_cosine(text_a, text_b):
    if not text_a.strip() or not text_b.strip():
        return 0.0
    vec = TfidfVectorizer(stop_words="english")
    try:
        matrix = vec.fit_transform([text_a, text_b])
    except ValueError:
        return 0.0  # zero shared vocabulary after stop-word removal
    return float(cosine_similarity(matrix[0], matrix[1])[0][0])


def judge_faithfulness_heuristic(question, answer, evidence_snippets):
    evidence_text = " ".join(evidence_snippets)
    sim = _tfidf_cosine(answer, evidence_text)
    return {"score": round(sim * 5, 2), "raw": f"tfidf_cosine={sim:.4f}", "method": "heuristic"}


def judge_answer_relevance_heuristic(question, answer):
    sim = _tfidf_cosine(question, answer)
    return {"score": round(sim * 5, 2), "raw": f"tfidf_cosine={sim:.4f}", "method": "heuristic"}

In [10]:
def score_faithfulness(question, answer, evidence_snippets):
    if USE_LLM_JUDGE:
        try:
            return judge_faithfulness_llm(question, answer, evidence_snippets)
        except Exception as e:
            print(f"[warn] LLM judge failed ({e}); falling back to heuristic for this call.")
    return judge_faithfulness_heuristic(question, answer, evidence_snippets)


def score_answer_relevance(question, answer):
    if USE_LLM_JUDGE:
        try:
            return judge_answer_relevance_llm(question, answer)
        except Exception as e:
            print(f"[warn] LLM judge failed ({e}); falling back to heuristic for this call.")
    return judge_answer_relevance_heuristic(question, answer)

## 6. Smoke test

In [12]:
_sample_question = "How do recent papers evaluate retrieval augmented generation?"
_sample_answer = (
    "Based on the selected evidence, recent papers evaluate retrieval augmented generation "
    "using counterfactual risk minimization and verbal reranking methods, measured against "
    "standard QA benchmarks."
)
_sample_evidence = [
    "Beyond Semantic Relevance: Counterfactual Risk Minimization for Robust Retrieval-Augmented Generation",
    "Verbal-R3: Verbal Reranker as the Missing Bridge between Retrieval and Reasoning",
]

faith = score_faithfulness(_sample_question, _sample_answer, _sample_evidence)
rel = score_answer_relevance(_sample_question, _sample_answer)

print("Faithfulness:", faith)
print("Answer relevance:", rel)

Faithfulness: {'score': 1.47, 'raw': 'tfidf_cosine=0.2934', 'method': 'heuristic'}
Answer relevance: {'score': 2.11, 'raw': 'tfidf_cosine=0.4222', 'method': 'heuristic'}


## Next step

`gold_qa_set.json` and `score_faithfulness()` / `score_answer_relevance()` are now reusable.
The next notebook runs each gold question through Member 1's `ask()` for the three mode presets
and the `WEIGHT_GRID` ablation sweep (fixed `max_total_evidence_chunks`), and assembles the
final comparison table: faithfulness, answer relevance, citation coverage, average latency, and
p95 latency per mode / weight combination.